# Notebook 2 — Your turn: Shakespeare's Sonnets + Ruff

Same workflow as **`01_demo_walkthrough.ipynb`** (notebook 1), rebuilt from scratch with a new dataset:
the 154 sonnets by Shakespeare (`datasets/sonnets.txt`, from the repository the lab suggests).

The second source stays the **Ruff FAQ**, so we can ask a multi-hop question that needs both.

| Lab task | Where |
|---|---|
| 1. Get your data | Step 1 |
| 2. Rebuild the vector store | Step 3 |
| 3. Rewrite the tool description | Step 5 |
| 4. Rebuild the agent | Step 6 |
| 5. Test: direct / Ruff-only / multi-hop | Steps 6–8 |
| 6. Reflect | last cell |


## Step 0 — Speed fix for this PC (same as notebook 1, Step 0)
Try IPv4 before IPv6 so each OpenAI call takes ~1 s instead of ~40 s.

In [1]:
# Fix for this PC: try IPv4 before IPv6, otherwise every OpenAI call waits ~40 s
# (safe to run more than once: the real function is saved only the first time)
import socket
if not hasattr(socket, "_original_getaddrinfo"):
    socket._original_getaddrinfo = socket.getaddrinfo
socket.getaddrinfo = lambda *args, **kwargs: sorted(
    socket._original_getaddrinfo(*args, **kwargs), key=lambda addr: addr[0] != socket.AF_INET
)

## Step 1 — Get the data
Download `sonnets.txt` once into the `datasets` folder (skipped if it is already there).

In [2]:
import os
import requests

SONNETS_URL = "https://raw.githubusercontent.com/martin-gorner/tensorflow-rnn-shakespeare/master/shakespeare/sonnets.txt"
sonnets_path = "./datasets/sonnets.txt"

if not os.path.exists(sonnets_path):
    with open(sonnets_path, "w", encoding="utf-8") as f:
        f.write(requests.get(SONNETS_URL, timeout=30).text)

with open(sonnets_path, encoding="utf-8") as f:
    print(f.read()[:700])

	SONNETS



TO THE ONLY BEGETTER OF
THESE INSUING SONNETS
MR. W. H. ALL HAPPINESS
AND THAT ETERNITY
PROMISED BY
OUR EVER-LIVING POET WISHETH
THE WELL-WISHING
ADVENTURER IN
SETTING FORTH
T. T.


I.

FROM fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light'st flame with self-substantial fuel,
Making a famine where abundance lies,
Thyself thy foe, to thy sweet self too cruel.
Thou that art now the world's fresh ornament
And only herald to the gaudy spring,
Within thine own bud buriest thy content
And, tender churl, makest 


## Step 2 — Imports, key and LLM (notebook 1, Steps 1–2)

In [3]:
os.environ.setdefault("USER_AGENT", "ironhack-lab")

from dotenv import load_dotenv, find_dotenv
from langchain_classic.agents import AgentType, Tool, initialize_agent
from langchain_classic.chains import RetrievalQA
from langchain_community.document_loaders import TextLoader, WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

## Step 3 — Sonnets vector store (notebook 1, Step 3)

**One change to the splitter:** every sonnet starts with its number on its own line (`XVIII.`).
Instead of cutting at any blank line, we cut only at a blank line that is followed by a Roman numeral line
(`is_separator_regex=True` lets the separator be a pattern). One sonnet is ~620 characters and two would be over 1000,
so each chunk ends up being **exactly one sonnet with its number** — the LLM can then say *which* sonnet it is quoting.

In [4]:
sonnet_documents = TextLoader(sonnets_path, encoding="utf-8").load()

sonnet_splitter = CharacterTextSplitter(
    separator=r"\n\n+(?=[IVXLC]+\.\n)",
    is_separator_regex=True,
    chunk_size=1000,
    chunk_overlap=0,
)
sonnet_texts = sonnet_splitter.split_documents(sonnet_documents)
print(len(sonnet_texts), "chunks = one per sonnet (the short dedication stays attached to sonnet I)")
print(sonnet_texts[18].page_content)  # chunk 18 = sonnet XIX (counting starts at 0)

154 chunks = one per sonnet (the short dedication stays attached to sonnet I)
XIX.

Devouring Time, blunt thou the lion's paws,
And make the earth devour her own sweet brood;
Pluck the keen teeth from the fierce tiger's jaws,
And burn the long-lived phoenix in her blood;
Make glad and sorry seasons as thou fleets,
And do whate'er thou wilt, swift-footed Time,
To the wide world and all her fading sweets;
But I forbid thee one most heinous crime:
O, carve not with thy hours my love's fair brow,
Nor draw no lines there with thine antique pen;
Him in thy course untainted do allow
For beauty's pattern to succeeding men.
  Yet, do thy worst, old Time: despite thy wrong,
  My love shall in my verse ever live young.


In [5]:
sonnets_db = InMemoryVectorStore.from_documents(sonnet_texts, embeddings)

sonnets = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=sonnets_db.as_retriever()
)

Quick check of the store alone (no LLM): which sonnet is closest to *"summer's day"*?

In [6]:
print(sonnets_db.similarity_search("compare thee to a summer's day", k=1)[0].page_content)

XVIII.

Shall I compare thee to a summer's day?
Thou art more lovely and more temperate:
Rough winds do shake the darling buds of May,
And summer's lease hath all too short a date:
Sometime too hot the eye of heaven shines,
And often is his gold complexion dimm'd;
And every fair from fair sometime declines,
By chance or nature's changing course untrimm'd;
But thy eternal summer shall not fade
Nor lose possession of that fair thou owest;
Nor shall Death brag thou wander'st in his shade,
When in eternal lines to time thou growest:
  So long as men can breathe or eyes can see,
  So long lives this and this gives life to thee.


## Step 4 — Ruff vector store (same as notebook 1, Step 4)

In [7]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
ruff_texts = text_splitter.split_documents(WebBaseLoader("https://docs.astral.sh/ruff/faq/").load())
ruff_db = InMemoryVectorStore.from_documents(ruff_texts, embeddings)

ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2209, which is longer than the specified 1000


Created a chunk of size 3190, which is longer than the specified 1000


Created a chunk of size 1017, which is longer than the specified 1000


Created a chunk of size 2313, which is longer than the specified 1000


## Step 5 — Tools with new descriptions (notebook 1, Step 5)

The description is what the agent routes on, so it says clearly *what is inside* the sonnets tool
(poems, love, beauty, time, the "dark lady"…) and that it is **not** about Python.

In [8]:
def ask_sonnets(question: str) -> str:
    return sonnets.invoke({"query": question})["result"]

def ask_ruff(question: str) -> str:
    return ruff.invoke({"query": question})["result"]

SONNETS_DESCRIPTION = (
    "useful for when you need to answer questions about Shakespeare's Sonnets "
    "(154 poems about love, beauty, time, ageing, a young man and a 'dark lady'). "
    "Input should be a fully formed question, not referencing any obscure pronouns from the conversation before."
)
RUFF_DESCRIPTION = (
    "useful for when you need to answer questions about ruff (a python linter and code formatter). "
    "Input should be a fully formed question, not referencing any obscure pronouns from the conversation before."
)

def make_tools(return_direct):
    return [
        Tool(name="Shakespeare Sonnets QA System", func=ask_sonnets,
             description=SONNETS_DESCRIPTION, return_direct=return_direct),
        Tool(name="Ruff QA System", func=ask_ruff,
             description=RUFF_DESCRIPTION, return_direct=return_direct),
    ]

## Step 6 — Agent as reasoner (`return_direct=False`) — notebook 1, Step 6

In [9]:
agent = initialize_agent(
    make_tools(return_direct=False), llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

C:\Users\Windows\AppData\Local\Temp\ipykernel_10552\1071011316.py:1: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(


**Test A — only the sonnets can answer:**

In [10]:
agent.invoke(
    "In the sonnet that begins 'Shall I compare thee to a summer's day', what does Shakespeare say will keep the beloved's beauty alive forever?"
)



> Entering new AgentExecutor chain...


I need to find out what Shakespeare says will preserve the beloved's beauty in the sonnet that begins with 'Shall I compare thee to a summer's day'. This is Sonnet 18, and I will look for specific references to beauty and immortality in that sonnet. 

Action: Shakespeare Sonnets QA System  
Action Input: What does Shakespeare say will keep the beloved's beauty alive forever in Sonnet 18?  


Observation: In Sonnet 18, Shakespeare suggests that the beloved's beauty will be kept alive forever through the lines of his poetry. He implies that as long as people read his verses, the beloved's beauty will continue to be celebrated and remembered, effectively granting it immortality.
Thought:

I now know the final answer.  
Final Answer: In Sonnet 18, Shakespeare suggests that the beloved's beauty will be kept alive forever through the lines of his poetry, implying that as long as people read his verses, the beloved's beauty will continue to be celebrated and remembered.

> Finished chain.


{'input': "In the sonnet that begins 'Shall I compare thee to a summer's day', what does Shakespeare say will keep the beloved's beauty alive forever?",
 'output': "In Sonnet 18, Shakespeare suggests that the beloved's beauty will be kept alive forever through the lines of his poetry, implying that as long as people read his verses, the beloved's beauty will continue to be celebrated and remembered."}

**Test B — only Ruff can answer:**

In [11]:
agent.invoke("Does Ruff support custom lint rules or plugins?")



> Entering new AgentExecutor chain...


I need to determine if Ruff, the Python linter and code formatter, supports custom lint rules or plugins. 
Action: Ruff QA System
Action Input: Does Ruff support custom lint rules or plugins?


Observation: No, Ruff does not support custom lint rules or plugins. It implements all rules natively and does not allow for third-party rules.
Thought:

I now know the final answer.
Final Answer: No, Ruff does not support custom lint rules or plugins. It implements all rules natively and does not allow for third-party rules.

> Finished chain.


{'input': 'Does Ruff support custom lint rules or plugins?',
 'output': 'No, Ruff does not support custom lint rules or plugins. It implements all rules natively and does not allow for third-party rules.'}

## Step 7 — Agent as router (`return_direct=True`) — notebook 1, Step 7
Same two questions; the agent should pick the same tools but stop right after the tool's answer.

In [12]:
router = initialize_agent(
    make_tools(return_direct=True), llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [13]:
router.invoke(
    "In the sonnet that begins 'Shall I compare thee to a summer's day', what does Shakespeare say will keep the beloved's beauty alive forever?"
)



> Entering new AgentExecutor chain...


I need to find out what Shakespeare says will preserve the beloved's beauty in the sonnet that begins with "Shall I compare thee to a summer's day." This is Sonnet 18, and I will look for specific references to beauty and immortality in that sonnet. 

Action: Shakespeare Sonnets QA System  
Action Input: What does Shakespeare say will keep the beloved's beauty alive forever in Sonnet 18?  


Observation: In Sonnet 18, Shakespeare suggests that the beloved's beauty will be kept alive forever through the lines of his poetry. He implies that as long as people read his verses, the beauty of his beloved will continue to live on, transcending time and decay.


> Finished chain.


{'input': "In the sonnet that begins 'Shall I compare thee to a summer's day', what does Shakespeare say will keep the beloved's beauty alive forever?",
 'output': "In Sonnet 18, Shakespeare suggests that the beloved's beauty will be kept alive forever through the lines of his poetry. He implies that as long as people read his verses, the beauty of his beloved will continue to live on, transcending time and decay."}

In [14]:
router.invoke("Does Ruff support custom lint rules or plugins?")



> Entering new AgentExecutor chain...


I need to determine if Ruff, the Python linter and code formatter, supports custom lint rules or plugins. 
Action: Ruff QA System
Action Input: Does Ruff support custom lint rules or plugins?


Observation: No, Ruff does not support custom lint rules or plugins. It implements all rules natively and does not allow for third-party rules.


> Finished chain.


{'input': 'Does Ruff support custom lint rules or plugins?',
 'output': 'No, Ruff does not support custom lint rules or plugins. It implements all rules natively and does not allow for third-party rules.'}

## Step 8 — Multi-hop question (needs both tools) — notebook 1, Step 8

Hop 1 (Ruff): the FAQ says which code formatter Ruff is designed to be compatible with → **Black**.
Hop 2 (Sonnets): Shakespeare uses that same word for his mistress in the "dark lady" sonnets.
The agent has to carry the word from the first answer into the second question.

In [15]:
agent.invoke(
    "Which code formatter is Ruff designed to be compatible with? "
    "Shakespeare uses that formatter's name as a colour word to describe his mistress in the sonnets: what does he say about her with it?"
)



> Entering new AgentExecutor chain...


I need to find out which code formatter Ruff is compatible with and then look for a reference in Shakespeare's sonnets that uses that formatter's name as a color word to describe his mistress. 

Action: Ruff QA System  
Action Input: "Which code formatter is Ruff designed to be compatible with?"  


Observation: Ruff is designed to be compatible with Black out-of-the-box.
Thought:

I now need to find a reference in Shakespeare's sonnets that uses "Black" as a color word to describe his mistress. 

Action: Shakespeare Sonnets QA System  
Action Input: "What does Shakespeare say about his mistress using the word 'Black'?"  


Observation: In Shakespeare's sonnets, particularly in Sonnet CXXVII, he discusses the concept of beauty and the color black. He suggests that black, which was not traditionally considered beautiful, has become associated with beauty in his time. He describes his mistress as having "raven black" brows and eyes that seem to mourn for those who are not born fair, implying that her black features are beautiful and worthy of admiration. 

In Sonnet CXXXII, he again refers to her eyes as "mourning" and suggests that her black features enhance her beauty, stating that beauty itself can be considered black. Overall, Shakespeare uses the word 'black' to challenge traditional notions of beauty and to elevate the beauty of his mistress, despite societal standards.
Thought:

I now know the final answer.  
Final Answer: Ruff is designed to be compatible with Black, and Shakespeare describes his mistress with "raven black" brows and eyes in Sonnet CXXVII, suggesting that her black features are beautiful and worthy of admiration, challenging traditional notions of beauty.

> Finished chain.


{'input': "Which code formatter is Ruff designed to be compatible with? Shakespeare uses that formatter's name as a colour word to describe his mistress in the sonnets: what does he say about her with it?",
 'output': 'Ruff is designed to be compatible with Black, and Shakespeare describes his mistress with "raven black" brows and eyes in Sonnet CXXVII, suggesting that her black features are beautiful and worthy of admiration, challenging traditional notions of beauty.'}

Same multi-hop question with the **router** — expected to stop after the first tool, so the second half stays unanswered:

In [16]:
router.invoke(
    "Which code formatter is Ruff designed to be compatible with? "
    "Shakespeare uses that formatter's name as a colour word to describe his mistress in the sonnets: what does he say about her with it?"
)



> Entering new AgentExecutor chain...


I need to find out which code formatter Ruff is compatible with and then look for a reference in Shakespeare's sonnets that uses that formatter's name as a color word to describe his mistress. 

Action: Ruff QA System  
Action Input: Which code formatter is Ruff designed to be compatible with?  


Observation: Ruff is designed to be compatible with Black out-of-the-box.


> Finished chain.


{'input': "Which code formatter is Ruff designed to be compatible with? Shakespeare uses that formatter's name as a colour word to describe his mistress in the sonnets: what does he say about her with it?",
 'output': 'Ruff is designed to be compatible with Black out-of-the-box.'}

## Reflection

_(written after running the cells above)_

**Did the agent pick the right tool each time?** Yes.
- *Test A (sonnets only)*: it chose **Shakespeare Sonnets QA System**, even recognised the line as Sonnet 18, and answered correctly — the poem itself ("So long lives this, and this gives life to thee") keeps the beauty alive.
- *Test B (Ruff only)*: it chose **Ruff QA System** → "No, Ruff implements all rules natively, no custom rules or plugins."
- *Multi-hop*: two actions in the right order — Ruff first (→ **Black**), then the sonnets with a self-contained question about "black" → Sonnet 127 ("my mistress' brows are raven black"; black is now "beauty's successive heir"). One combined final answer.

The clear tool descriptions made routing easy: the two sources have nothing in common, so there was never a doubt.

**`return_direct=False` vs `True`:**
- *False (reasoner)*: after the tool's *Observation* the agent takes one more *Thought* and writes its own *Final Answer* — an extra LLM call, but it can chain several tools.
- *True (router)*: same tool choices, but it stops right after the first tool and returns that answer word for word — faster and cheaper.
- On the multi-hop question the router **only answered the first half** (Black) and never asked the sonnets tool. So a router is good when each question belongs to exactly one source; questions that combine sources need the reasoner.

**Dataset-specific lesson:** splitting at the Roman numeral lines (one sonnet per chunk) made the retrieved context clean, so the answers could name the right sonnet.
